# Trajectory Optimization: Shortest Reliable Path

**Primary principle:**
THE SHORTEST RUN IS NOT NECESSARILY THE BEST RUN.
OPTIMIZE COST / LATENCY / WORK
SUBJECT TO
QUALITY + SAFETY + GROUNDING CONSTRAINTS.

This notebook demonstrates how to analyze agent trajectories for unnecessary work, remove duplicate reads, apply safe caching, parallelize independent tools, and bound retries and reflections, all without violating the evaluation boundaries established in Course 05.

## Part 1: Baseline Typed Trajectory

We start with a baseline trajectory from the Northstar EU checkout latency scenario. The baseline has unnecessary work: it reads health twice, logs twice, and runs everything sequentially.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("curriculum/intermediate/06-trajectory-optimization"))

from policy import (
    StepType, ResultStatus, ToolEffect, StepClassification,
    OptimizationType, ToolDefinition, TrajectoryStep, Trajectory,
    TrajectoryMetrics, OptimizationCandidate, OptimizationPlan,
    OptimizationResult, TrajectoryComparison,
    classify_steps, can_parallelize, is_valid_cache_hit, compute_metrics, optimization_regression_gate
)

tools = {
    "get_service_health": ToolDefinition(name="get_service_health", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="db"),
    "query_logs": ToolDefinition(name="query_logs", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="logs"),
    "get_deployment": ToolDefinition(name="get_deployment", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="k8s"),
    "restart_service": ToolDefinition(name="restart_service", effect=ToolEffect.WRITE, supports_parallel=False, cacheable=False)
}

baseline = Trajectory(
    run_id="run-baseline-01",
    tenant_id="northstar",
    final_answer="Identify database connection pool exhaustion in EU-West.",
    agent_version="1.0", prompt_version="1.0", model_version="1.0", tool_version="1.0", policy_version="1.0", dataset_version="1.0",
    steps=[
        TrajectoryStep(step_id="1", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=500, cost_usd=0.01),
        TrajectoryStep(step_id="2", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="3", step_type=StepType.TOOL_CALL, tool_name="query_logs", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=1200, cost_usd=0.02),
        TrajectoryStep(step_id="4", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="5", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=500, cost_usd=0.01),
        TrajectoryStep(step_id="6", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="7", step_type=StepType.TOOL_CALL, tool_name="query_logs", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=1200, cost_usd=0.02),
        TrajectoryStep(step_id="8", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="9", step_type=StepType.TOOL_CALL, tool_name="get_deployment", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=600, cost_usd=0.01),
        TrajectoryStep(step_id="10", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
    ]
)
print("Baseline trajectory loaded.")

## Part 2 & 3: Instrumentation, Metrics, and Step Classification

Before optimizing, we must classify each step. Is this a `DUPLICATE_READ`, a `SIDE_EFFECT`, or `REQUIRED_EVIDENCE`?

In [ ]:
classified_baseline = classify_steps(baseline, tools)

print("Step Classifications:")
for s in classified_baseline.steps:
    if s.step_type == StepType.TOOL_CALL:
        print(f"  {s.tool_name} -> {s.classification.value}")


## Part 4: Duplicate Removal

We see duplicate reads. Can we just remove them? Yes, because they are `DUPLICATE_READ`. If they were a repeated WRITE, we would NOT remove them as a simple optimization—that would mask an idempotency bug!

In [ ]:
# Create optimized candidate by stripping duplicates
optimized_steps = [s for s in classified_baseline.steps if s.classification != StepClassification.DUPLICATE_READ and not (s.step_type == StepType.TOOL_RESULT and getattr(s, '_was_dup', False))]

# To keep the example simple, we also manually filter the tool results for the dups.
# Let's just define the optimized trajectory explicitly:
optimized_steps_clean = [
    TrajectoryStep(step_id="1", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=500, cost_usd=0.01),
    TrajectoryStep(step_id="2", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
    TrajectoryStep(step_id="3", step_type=StepType.TOOL_CALL, tool_name="query_logs", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=1200, cost_usd=0.02),
    TrajectoryStep(step_id="4", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
    TrajectoryStep(step_id="9", step_type=StepType.TOOL_CALL, tool_name="get_deployment", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=600, cost_usd=0.01),
    TrajectoryStep(step_id="10", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, latency_ms=10, cost_usd=0),
]


## Part 5: Dependency-Aware Parallelism

Can we parallelize the independent reads? Total work vs wall-clock latency.

In [ ]:
s1 = optimized_steps_clean[0] # health
s2 = optimized_steps_clean[2] # logs
s3 = optimized_steps_clean[4] # deploy

print(f"Can parallelize health and logs? {can_parallelize(s1, s2, tools)}")

# Sequential: 500 + 1200 + 600 = 2300ms
# Parallel: max(500, 1200, 600) = 1200ms


## Part 6: Caching

Cache safe deterministic reads only.

In [ ]:
cache_entry = {"policy_version": "1.0", "expires_at": 999999999}
s = TrajectoryStep(step_id="1", step_type=StepType.TOOL_CALL, tool_name="get_service_health", target_tenant_id="northstar", latency_ms=10, cost_usd=0, observed_at=100)

print("Valid cache hit?", is_valid_cache_hit(s, cache_entry, "northstar", "1.0"))
print("Valid cache hit (cross-tenant)?", is_valid_cache_hit(s, cache_entry, "globex", "1.0"))


## Part 7 & 8 & 9: Reflection Limits, Early Stopping, Retry Semantics

- **Reflection:** Bound open-ended loops. Use threshold-based early stopping.
- **Early Stopping:** If evidence `{'health', 'logs'}` is required, stop retrieving once it's met. Don't fetch `deployment` just because budget remains.
- **Retries:** Only retry TIMEOUT or REPAIRABLE schema errors. Do NOT retry POLICY_BLOCKED or AUTH_BLOCKED.

## Part 10: Baseline vs Optimized Comparison

Let's compute the trajectory metrics delta.

In [ ]:
optimized = Trajectory(
    run_id="run-optimized-01",
    tenant_id="northstar",
    final_answer="Identify database connection pool exhaustion in EU-West.",
    agent_version="1.0", prompt_version="1.0", model_version="1.0", tool_version="1.0", policy_version="1.0", dataset_version="1.0",
    steps=optimized_steps_clean
)

comparison = compute_metrics(baseline, optimized)
print(f"Delta Wall Clock Latency: {comparison.delta_wall_clock_latency_ms}ms")
print(f"Delta Cost: ${comparison.delta_cost_usd}")


## Part 11: Bad Optimization / Regression Rejection

A faster unsafe trajectory MUST FAIL the regression gate.

In [ ]:
# Candidate drops a required evidence log fetch to save 1200ms, making the answer ungrounded.
comparison.candidate.required_evidence_recall = 0.5 
comparison.delta_required_evidence_recall = -0.5

print("Does bad optimization pass the regression gate?", optimization_regression_gate(comparison))


## Part 12: Pareto Trade-offs

Lowest latency, lowest cost, and highest reliability might be three completely different execution paths. Accept that optimization is a Pareto frontier, not a single global maximum.

## Part 13: Optional Deep Dive: DSPy Program Optimization

We can use programmatic compilers like DSPy to optimize the prompts themselves to encourage the model toward the optimized trajectory shape, but DSPy alone is NOT trajectory optimization—it is LM program optimization against a metric.

In [ ]:
# 1. Define DSPy Signature
# class OptimizerSignature(dspy.Signature):
#     ...
#
# 2. Define Teleprompter
# from dspy.teleprompt import BootstrapFewShot
# ...
